In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

from xgboost import XGBClassifier

In [4]:
df = pd.read_excel("customer_intent_classification_dataset.xlsx")

# Keep only required columns
df = df[['Customer_Utterance', 'Intent_Label']]

# Remove missing values
df.dropna(inplace=True)

df.head()

,Customer_Utterance,Intent_Label
0,I'd like to understand the full scope before d...,Neutral
1,Can you confirm the price? I'm ready to place ...,Strongly Interested
2,I've compared all options and I want this one....,Strongly Interested
3,This seems like a good fit. What are the next ...,Interested
4,I've compared all options and I want this one....,Strongly Interested


In [5]:
le = LabelEncoder()
df['Intent_Label'] = le.fit_transform(df['Intent_Label'])

In [6]:
tfidf = TfidfVectorizer(max_features=2000, ngram_range=(1,2))

X = tfidf.fit_transform(df['Customer_Utterance']).toarray()
y = df['Intent_Label']

In [7]:
tfidf = TfidfVectorizer(max_features=2000, ngram_range=(1,2))

X = tfidf.fit_transform(df['Customer_Utterance']).toarray()
y = df['Intent_Label']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [11]:
model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    eval_metric='mlogloss'
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [12]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.9711538461538461

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        15
           1       1.00      1.00      1.00        22
           2       0.89      1.00      0.94        24
           3       1.00      0.86      0.92        21
           4       1.00      1.00      1.00        22

    accuracy                           0.97       104
   macro avg       0.98      0.97      0.97       104
weighted avg       0.97      0.97      0.97       104



In [15]:
def predict():
    utterance = input("Enter customer utterance: ")

    # Convert text
    text_vec = tfidf.transform([utterance]).toarray()

    # Get probability scores
    probs = model.predict_proba(text_vec)[0]

    # Get predicted class
    pred_class = np.argmax(probs)
    pred_label = le.inverse_transform([pred_class])[0]

    print("\n Predicted Intent:", pred_label)
    print("\n Intent Scores:")

    for i, prob in enumerate(probs):
        label = le.inverse_transform([i])[0]
        print(f"{label}: {prob:.4f}")

In [21]:
predict()

Enter customer utterance: I'd like to understand the full scope

 Predicted Intent: Strongly Interested

 Intent Scores:
Casually Browsing: 0.0126
Interested: 0.1441
Neutral: 0.1114
Not Interested: 0.0233
Strongly Interested: 0.7086
